# Notebook Set-Up

In [ ]:
# # FOR COLAB
# # 1. run cell
# # 2. restart run time, comment cell out (DO NOT RUN AFTER RESTARTING)
# # uninstall any existing torch
# !pip uninstall -y torch torchvision torchaudio torch-geometric torch-scatter torch-sparse torch-cluster torch-spline-conv

# # install fairchem
# !pip install git+https://github.com/facebookresearch/fairchem.git@fairchem_core-2.0.0#subdirectory=packages/fairchem-core

# # import torch
# import torch
# TORCH_VER = torch.__version__.split('+')[0]
# CUDA_VER = 'cu' + torch.version.cuda.replace('.', '')
# print(f"Detected: torch={TORCH_VER} {CUDA_VER}")

# # install PyG wheel
# !pip install torch-scatter torch-sparse torch-cluster torch-spline-conv \
#   -f https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html

# # install torch_geometric
# !pip install torch_geometric

In [1]:
# check that installs worked
import torch, torch_geometric, fairchem.core
from torch_cluster import radius_graph
print("Torch:", torch.__version__)
print("PyG:", torch_geometric.__version__)
print("Fairchem:", fairchem.core.__version__)
print("radius_graph: OK")

Torch: 2.6.0+cu124
PyG: 2.7.0
Fairchem: 2.0.0
radius_graph: OK


In [4]:
# get repo (colab)
!git clone https://github.com/yasheshak/Chem-277B-Final-Project.git

Cloning into 'Chem-277B-Final-Project'...
remote: Enumerating objects: 188, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 188 (delta 13), reused 11 (delta 10), pack-reused 171 (from 1)
Receiving objects: 100% (188/188), 2.47 MiB | 22.18 MiB/s, done.
Resolving deltas: 100% (92/92), done.


In [5]:
# open directory (colab)
%cd Chem-277B-Final-Project

/content/Chem-277B-Final-Project/Chem-277B-Final-Project


In [6]:
# update repo (colab)
!git pull

Already up to date.


In [7]:
# directory check (colab)
!ls -a

.			    extract.py		      read_multi_ase.py
..			    .git		      SchNet_for_import.py
db_explore.ipynb	    .gitignore		      SchNet_GNN_Baseline.ipynb
DimeNet_baseline_new.ipynb  hyperparam_test.ipynb     SchNet_GNN.ipynb
.DS_Store		    hyperparam_test_v2.ipynb  SchNet_Normalize.ipynb
EDA			    Makefile		      simpleGNN
EDA.ipynb		    __pycache__
environment.yml		    README.md


In [8]:
# sync google drive (colab)
from google.colab import auth
auth.authenticate_user()

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# system imports
import os
import sys
import time
sys.path.append('/content/Chem-277B-Final-Project')

# local imports
from read_multi_ase import *
from extract import *
from SchNet_for_import import *

import glob
from typing import Union

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import random_split

# pytorch
from torch_geometric.data import Data
from torch_geometric.nn import SchNet
from torch_geometric.loader import DataLoader

# fairchem
from fairchem.core.datasets import AseDBDataset

ModuleNotFoundError: No module named 'fairchem'

# Baseline Sweep

In [10]:
# ---------------------------------- Initialize hyperparam range -------------------------------
# parameters to optimize
''' Update dictionaries with intended values to test'''

# hidden_channels --> length of each atom’s representative feature vector
hidden_channels = {
    'name': 'hidden_channels',
    'start': 64,
    'end': 256,
    'step': 64
}
# num_filters --> number of output channels / size of filter generating network
num_filters = {
    'name': 'num_filters',
    'start': 64,
    'end': 256,
    'step': 64
}
# num_interactions --> number of message passing operations per forward pass
num_interactions = {
    'name': 'num_interactions',
    'start': 4,
    'end': 8,
    'step': 1
}
# num_gaussians --> number of radial basis functions for encoding resolution for atomic distances
num_gaussians = {
    'name': 'num_gaussians',
    'start': 25,
    'end': 100,
    'step': 25
}
# cutoff # 4 - 12
cutoff = {
    'name': 'cutoff',
    'start': 4,
    'end': 8,
    'step': 1
}
# max_num_neighbors
max_num_neighbors = {
    'name': 'max_num_neighbors',
    'start': 20,
    'end': 50,
    'step': 10
}

# store dictionaries in a list for baseline_sweep func. example:
# params_to_optimize = [hidden_channels, num_filters, num_interactions, num_gaussians, cutoff, max_num_neighbors]

# dictionary for unchanging parameters. during param_tuning, the target param's value will be reassigned
default_params = {
'hidden_channels': 128,
'num_filters': 128,
'num_interactions': 6,
'num_gaussians': 50,
'cutoff': 5,
'max_num_neighbors': 32
}

In [ ]:
# ------------------------------- Baseline Sweep Function -------------------------------------------
def param_tuning(param: str,
                 param_start: int,
                 param_end: int,
                 param_step: int,
                 file_path_to_save: str,
                 train_data,
                 val_data,
                 test_data):
    
    ''' 
    For each parameter dict object, initialize values, and iterate through. param_tuning is called for each parameter.

    Parameters
    -------------------------------
    param : str
    parameter name
    param_start : int
        start of range of values to test
    param_end : int
        end of range of values to test
    param_step : int
        step size for np.arange initialization
    file_path_to_save : str
        filepath for loss saving graphs
    train 
        train loader
    val
        val loader
    test 
        test loader

    Returns
    ---------------------------------
    List of dictionaries for dataframe conversion. Saves graphs to filepath
    
    '''

    # initialize numpy array for param vals
    param_vals = np.arange(start=param_start, stop=(param_end+1), step=param_step)

    # copy default_params dict into params
    params = default_params.copy()

    # save params and results in a list (for future dataframe --> excel conversion)
    rows = []
    for val in param_vals:
      # update default_param dict with target value
      params[param] = val

      device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

      # initialize model with desired parameters using updated params dictionary
      model = SchNetModel(
          params['hidden_channels'],
          params['num_filters'],
          params['num_interactions'],
          params['num_gaussians'],
          params['cutoff'],
          params['max_num_neighbors'],
          readout = "add",
          dipole = False,
          mean = None,
          std = None,
          atomref = None,
          train_mean = None,
          train_std = None
      ).to(device)

      optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)
      loss_function = torch.nn.SmoothL1Loss()

      # run model
      epochs = 50
      train_losses = np.zeros(epochs)
      val_losses = np.zeros(epochs)

      start_time = time.time()
      for epoch in range(epochs):
          train_loss = train(model, train_data, device, optimizer, loss_function)
          val_loss = evaluate(model, val_data, device, loss_function)

          train_losses[epoch] = train_loss
          val_losses[epoch] = val_loss

      stop_time = time.time()
      runtime = stop_time - start_time

      print(f'hyperparameter: {param}, value: {val}')

      # save photo of Loss graph
      graph = plot_losses(train_losses, val_losses)
      filename = f'{param}_{val}.png'
      filepath = os.path.join(file_path_to_save, filename)
      plt.savefig(filepath)
      plt.close()

      # Get RMSE + MAE
      mae, rmse = test(model, test, device, loss_function)
      print(f'{mae}, {rmse}')

      # save as row in df
      row = {
          'tested_param': param,
          'value': val,
          'hidden_channels': params['hidden_channels'],
          'num_filters': params['num_filters'],
          'num_interactions': params['num_interactions'],
          'num_gaussians': params['num_gaussians'],
          'cutoff': params['cutoff'],
          'max_num_neighbors': params['max_num_neighbors'],
          'MAE': mae,
          'RMSE': rmse,
          'runtime': runtime}

      rows.append(row) # append dictionary to rows list

    return rows # return list of dictionaries. one list per hyperparameter


In [ ]:
# --------------------------------- Baseline Sweep function --------------------------------------
def baseline_sweep(params_to_optimize: list, filename: str, file_path_to_save, train, val, test):
  ''' 
  Iterate through each parameter in params_to_optimize list

  Parameters
  -----------------------------------------
  params_to_optimize : list
    list of parameter dictionary objects
  filename : str
    output name of excel file
  train 
    train loader
  val
    val loader
  test 
    test loader
  
  Returns
    ----------------------------------
    List of dictionaries, storing the results and its respective parameters
  
  '''
  full_data = [] # store results

  # iterate through each parameter
  for param in params_to_optimize:
    results = param_tuning(param['name'], param['start'], param['end'], param['step'], file_path_to_save, train, val, test) # iterate through range of values for param
    full_data.extend(results) # adds each dictionary from results list to full_data

  df = pd.DataFrame(full_data)
  filepath = os.path.join(file_path_to_save, f"{filename}.xlsx")
  df.to_excel(filepath, index=False) # save dataframe as excel for viewing

# Running baseline sweep

In [14]:
# ============================= Initialize training set =============================
''' Update path to dataset '''
dataset_path = '/content/drive/MyDrive/train_4M/data0000.aselmdb' # given the local dataset path, loads the first .aselmdb file
files_list = dataset_path

''' update num of molecules'''
dataset = process_file(files_list, molecule_type = 'biomolecules', max_molecules = 500)

# convert to torch
torch_data = get_data(dataset)
train_dataset, val_dataset, test_dataset = split_data(torch_data, 0.2, 0.2)

# load
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

Processed 500 atoms


In [ ]:
# ================================== Initialize device, optimizer, and loss function =================================
#Determine the device to be used
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#Initialize model with desired parameters
model = SchNetModel().to(device)
#Create ADAM optimizer based on model's parameters and desired learning rate
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)
#Select loss function for model
loss_function = torch.nn.SmoothL1Loss()

In [ ]:
# ================================================ Run baseline sweep =============================================
params_to_optimize = [hidden_channels, num_filters, num_interactions, num_gaussians, cutoff, max_num_neighbors]

''' update filename for excel sheet AND path to save graphs + excel sheet '''
baseline_sweep(params_to_optimize = params_to_optimize,
               filename = 'test_1',
               file_path_to_save = '/content/drive/MyDrive/hyperparam_tuning',
               train = train_loader,
               val = val_loader,
               test = test_loader)

hyperparameter: hidden_channels, value: 64
3.0478297233581544, 3.9908994289067157
hyperparameter: hidden_channels, value: 128
3.0141558933258055, 3.795892985629285
hyperparameter: hidden_channels, value: 192
2.91520299911499, 3.7890937171515273
hyperparameter: hidden_channels, value: 256
2.918353309631348, 3.761345703388181
hyperparameter: num_filters, value: 64
3.128122787475586, 3.9277773172922927
hyperparameter: num_filters, value: 128
3.660849609375, 4.917663376639102
hyperparameter: num_filters, value: 192
2.7362413597106934, 3.4361302369007856
hyperparameter: num_filters, value: 256
3.0165115356445313, 3.6502859395519582
hyperparameter: num_interactions, value: 4
3.555314874649048, 4.65864589418243
hyperparameter: num_interactions, value: 5
2.939188528060913, 3.78771431193831
hyperparameter: num_interactions, value: 6
3.468201913833618, 4.2070780150241465
hyperparameter: num_interactions, value: 7
2.729451813697815, 3.4240074668695906
hyperparameter: num_interactions, value: 8
3.

 # Gridsearch

In [ ]:
# ============================= Initialize training set =============================
''' Update path to dataset '''
dataset_path = '/content/drive/MyDrive/train_4M/data0000.aselmdb' # given the local dataset path, loads the first .aselmdb file
files_list = dataset_path

''' update num of molecules'''
dataset = process_file(files_list, molecule_type = 'biomolecules', max_molecules = 500)

# convert to torch
torch_data = get_data(dataset)
train_dataset, val_dataset, test_dataset = split_data(torch_data, 0.2, 0.2)

# load
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

# ================================== Initialize device, optimizer, and loss function =================================
#Determine the device to be used
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#Initialize model with desired parameters
model = SchNetModel().to(device)
#Create ADAM optimizer based on model's parameters and desired learning rate
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)
#Select loss function for model
loss_function = torch.nn.SmoothL1Loss()

In [ ]:
# ------------------------ Target Parameters -------------------------
hidden_channels_param = [128, 192, 256] # 0
num_filters_param = [128, 192, 256] # 1
num_interactions_param = [6, 7, 8] # 2
num_gaussians_param = [25, 50, 75] # 3
cutoff_param = [4, 5, 6, 7] # 4
max_num_neighbors_param = [30, 40] # 5

# list of parameters
chosen_params = [hidden_channels_param, num_filters_param, num_interactions_param, num_gaussians_param, cutoff_param, max_num_neighbors_param]

parameter_grids = np.meshgrid(*chosen_params, indexing='ij')
all_param_combos = np.column_stack([g.ravel() for g in parameter_grids]) # get np.array of parameter combinations. 1 row = 1 combination

print(all_param_combos.shape)

(648, 6)


In [5]:
all_param_combos

array([[128, 128,   6,  25,   4,  30],
       [128, 128,   6,  25,   4,  40],
       [128, 128,   6,  25,   5,  30],
       ...,
       [256, 256,   8,  75,   6,  40],
       [256, 256,   8,  75,   7,  30],
       [256, 256,   8,  75,   7,  40]], shape=(648, 6))

In [ ]:
# ------------------------------- Gridsearch Function -------------------------------------------
def param_gridsearch(parameters: np.array,
                 file_path_to_save: str,
                 train_data,
                 val_data,
                 test_data):
    ''' 
    Iterate through every possible parameter combination

    Parameters
    ---------------------------------
    parameters : np.array
        array of all parameter combinations
    filename : str
        output name of excel file
    train 
        train loader
    val
        val loader
    test 
        test loader

    Returns
    ----------------------------------
    List of dictionaries, storing the results and its respective parameters
    '''
    rows = [] # save params and results in a list (for future dataframe --> excel conversion)
    for row_of_params in parameters:
      
      params = row_of_params.tolist()

      # initialize device
      device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

      # initialize model with desired parameters using updated params dictionary
      model = SchNetModel(
          params[0],
          params[1],
          params[2],
          params[3],
          params[4],
          params[5],
          readout = "add",
          dipole = False,
          mean = None,
          std = None,
          atomref = None,
          train_mean = None,
          train_std = None
      ).to(device)

      optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)
      loss_function = torch.nn.SmoothL1Loss()

      # run model
      epochs = 50
      train_losses = np.zeros(epochs)
      val_losses = np.zeros(epochs)

      start_time = time.time()
      for epoch in range(epochs):
          train_loss = train(model, train_data, device, optimizer, loss_function)
          val_loss = evaluate(model, val_data, device, loss_function)

          train_losses[epoch] = train_loss
          val_losses[epoch] = val_loss

      stop_time = time.time()
      runtime = stop_time - start_time

      print(f'Parameters: {row_of_params}')

      # save photo of Loss graph
      graph = plot_losses(train_losses, val_losses)
      filename = f'{params}.png'
      filepath = os.path.join(file_path_to_save, filename)
      plt.savefig(filepath)
      plt.close()

      # Get RMSE + MAE
      mae, rmse = test(model, test_loader, device, loss_function)
      print(f'{mae}, {rmse}')

      # save as row in df
      row = {
          'hidden_channels': params[0],
          'num_filters': params[1],
          'num_interactions': params[2],
          'num_gaussians': params[3],
          'cutoff': params[4],
          'max_num_neighbors': params[5],
          'MAE': mae,
          'RMSE': rmse,
          'runtime': runtime}

      rows.append(row) # append dictionary to rows list

    return rows # return list of dictionaries. one list per hyperparameter


In [ ]:
# --------------------------------- Grid Search Saving Results function --------------------------------------
def save_grid_search(parameters: np.array, filename: str, file_path_to_save, train, val, test):
  '''
  Run's grid search on preset parameter values

  Parameters
  ---------------------------------
  parameters : np.array
    array of all parameter combinations
  filename : str
    output name of excel file
  train 
    train loader
  val
    val loader
  test 
    test loader

  Returns
  ----------------------------------
  excel sheet with results and corresponding parameters saved to designated filepath
  '''
  full_data = [] # store results

  results = param_gridsearch(parameters, 
                             file_path_to_save, 
                             train, 
                             val, 
                             test)
  
  full_data.extend(results) # adds each dictionary from results list to full_data

  df = pd.DataFrame(full_data)
  filepath = os.path.join(file_path_to_save, f"{filename}.xlsx")
  df.to_excel(filepath, index=False) # save dataframe as excel for viewing

# Running Grid Search

In [ ]:
save_grid_search(all_param_combos, 
                'gridsearch', 
                file_path_to_save = '/content/drive/MyDrive/hyperparam_tuning',
                train_data = train_loader,
                val_loader = val_loader,
                test_data = test_loader)